In [1]:
from copy import deepcopy
from datetime import date
from pathlib import Path

import numpy as np

from conf.behavior_cloning.diffusion.five_demos.default import config
from rlbench.action_modes.arm_action_modes import BimanualEndEffectorPoseViaIK
from tapas_gmm.encoder.encoder import ObservationEncoderConfig
from tapas_gmm.env.rlbench import RLBenchEnvironment, RLBenchEnvironmentConfig
from tapas_gmm.policy.diffusion import DiffusionPolicy
from tapas_gmm.utils.select_gpu import device

import imageio.v2 as imageio

2026-07-21 13:16:15.156 | INFO     |  Running on cpu


In [2]:
joint_config = deepcopy(config.policy)
leader_config = deepcopy(config.policy)
follower_config = deepcopy(config.policy)

for policy_config in (joint_config, leader_config, follower_config):
    policy_config.obs_dim = 35
    policy_config.horizon = 16
    policy_config.n_obs_steps = 2
    policy_config.n_action_steps = 8
    policy_config.training = None
    policy_config.obs_encoder = ObservationEncoderConfig(
        ee_pose=True,
        object_poses=True,
    )
    policy_config.unet.down_dims = (64, 128, 256)

joint_config.action_dim = 16
joint_config.unet.input_dim = 16
joint_config.unet.global_cond_dim = 35 * 2

leader_config.action_dim = 8
leader_config.arm = "left"
leader_config.unet.input_dim = 8
leader_config.unet.global_cond_dim = 35 * 2

follower_config.action_dim = 8
follower_config.arm = "right"
follower_config.condition_on_arm = "left"
follower_config.unet.input_dim = 8
follower_config.unet.global_cond_dim = 35 * 2 + 16 * 8

In [3]:
checkpoint_root = Path("../artifacts/checkpoints/diffusion")
joint_dir = max(path for path in (checkpoint_root / "joint").iterdir() if path.is_dir())
leader_follower_dir = max(
    path for path in (checkpoint_root / "leader_follower").iterdir() if path.is_dir()
)

joint_policy = DiffusionPolicy(joint_config).to(device)
leader_policy = DiffusionPolicy(leader_config).to(device)
follower_policy = DiffusionPolicy(follower_config).to(device)

joint_policy.from_disk(str(joint_dir / "latest.pt"))
leader_policy.from_disk(str(leader_follower_dir / "leader/latest.pt"))
follower_policy.from_disk(str(leader_follower_dir / "follower/latest.pt"))

joint_policy.eval()
leader_policy.eval()
follower_policy.eval()

2026-07-21 13:16:23.260 | INFO     |  Initializing DiffusionPolicy:
2026-07-21 13:16:23.261 | INFO     |    Initializing Policy:
2026-07-21 13:16:23.503 | INFO     |    number of parameters: 5525968
2026-07-21 13:16:23.764 | INFO     |    No encoder config provided. Using None.
None
2026-07-21 13:16:23.814 | INFO     |    number of parameters: 5522376
None
2026-07-21 13:16:23.862 | INFO     |    number of parameters: 5981128
None


DiffusionPolicy(
  (model): ConditionalUnet1D(
    (mid_modules): ModuleList(
      (0-1): 2 x ConditionalResidualBlock1D(
        (blocks): ModuleList(
          (0-1): 2 x Conv1dBlock(
            (block): Sequential(
              (0): Conv1d(256, 256, kernel_size=(5,), stride=(1,), padding=(2,))
              (1): GroupNorm(8, 256, eps=1e-05, affine=True)
              (2): Mish()
            )
          )
        )
        (cond_encoder): Sequential(
          (0): Mish()
          (1): Linear(in_features=454, out_features=512, bias=True)
          (2): Rearrange('batch t -> batch t 1')
        )
        (residual_conv): Identity()
      )
    )
    (diffusion_step_encoder): Sequential(
      (0): SinusoidalPosEmb()
      (1): Linear(in_features=256, out_features=1024, bias=True)
      (2): Mish()
      (3): Linear(in_features=1024, out_features=256, bias=True)
    )
    (up_modules): ModuleList(
      (0): ModuleList(
        (0): ConditionalResidualBlock1D(
          (blocks): M

In [4]:
tapas_env = RLBenchEnvironment(
    RLBenchEnvironmentConfig(
        action_mode=BimanualEndEffectorPoseViaIK,
        robot_setup="dual_panda",
        task="BimanualDualPushButtons",
        cameras=("front",),
        camera_pose={},
        image_size=(128, 128),
        static=False,
        headless=False,
        scale_action=False,
        delay_gripper=False,
        gripper_plot=False,
        absolute_action_mode=True,
        action_frame="world",
    )
)

In [5]:
def predict_joint(obs):
    trajectory, _ = joint_policy.predict(obs)
    return np.concatenate((trajectory.ee, trajectory.gripper), axis=-1)


def predict_leader_follower(obs):
    leader_trajectory, leader_info = leader_policy.predict(obs)
    follower_trajectory, _ = follower_policy.predict(
        obs,
        condition=leader_info["action_pred"],
    )
    return np.concatenate(
        (
            leader_trajectory.ee,
            follower_trajectory.ee,
            leader_trajectory.gripper[:, None],
            follower_trajectory.gripper[:, None],
        ),
        axis=-1,
    )

In [6]:
def run_episode(architecture, max_steps=200):
    obs = tapas_env.reset()
    joint_policy.reset_episode(tapas_env)
    leader_policy.reset_episode(tapas_env)
    follower_policy.reset_episode(tapas_env)

    total_reward = 0
    step = 0
    done = False
    frames = []

    while step < max_steps and not done:
        if architecture == "joint":
            actions = predict_joint(obs)
        else:
            actions = predict_leader_follower(obs)

        for action in actions:
            obs, reward, done, _ = tapas_env.step(action)
            total_reward += reward
            step += 1

            if obs is not None:
                frame = obs.cameras["front"].rgb

                if frame.ndim == 4:
                    frame = frame[0]

                if frame.shape[0] == 3:
                    frame = frame.permute(1, 2, 0)

                frame = frame.clamp(0, 1).mul(255).byte().cpu().numpy()
                frames.append(frame)

            if done or obs is None or step >= max_steps:
                break

    video_dir = Path("../artifacts/videos/diffusion") / architecture / date.today().isoformat()
    video_dir.mkdir(parents=True, exist_ok=True)
    imageio.mimsave(video_dir / "run.mp4", frames, fps=20)

    print("architecture:", architecture)
    print("steps:", step)
    print("total_reward:", total_reward)

In [7]:
run_episode("joint")

[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.


2026-07-21 13:16:40.539 | INFO     |  Action [ 0.2369685  -0.24360763  1.47221589 -0.98297077  0.09983229  0.12583555
 -0.08926004  1.          0.          0.29337558  0.06848705  1.47706914
  0.00379554  0.99317181 -0.04300052  0.10837952  1.          0.        ]
2026-07-21 13:16:42.017 | INFO     |  Action [ 2.37440810e-01 -2.36231625e-01  1.47221589e+00 -9.67970192e-01
 -1.00752630e-03  1.20281510e-01 -2.20374867e-01  1.00000000e+00
  0.00000000e+00  2.82157362e-01  6.74929768e-02  1.47706914e+00
  6.72993138e-02  9.84718919e-01  1.05960041e-01  1.20713957e-01
  1.00000000e+00  0.00000000e+00]
2026-07-21 13:16:42.325 | INFO     |  Action [ 0.25503772 -0.24707903  1.47221589 -0.98938286  0.05080388  0.12741224
 -0.04802851  1.          0.          0.29892164  0.07253002  1.47706914
  0.00330642  0.99430048  0.00482248  0.10645323  1.          0.        ]
2026-07-21 13:16:42.816 | INFO     |  Action [ 0.23908772 -0.2392274   1.46960163 -0.9835127  -0.02602993  0.11983459
 -0.13290983 

2026-07-21 13:16:46.105 | INFO     |  Action [ 0.25720632 -0.24146266  1.47221589 -0.99029094 -0.04608886  0.12774841
 -0.02966438  1.          0.          0.29861757  0.07742172  1.47706914
 -0.01638656  0.99467432 -0.03061314  0.09704214  1.          0.        ]
2026-07-21 13:16:46.526 | INFO     |  Action [ 0.22351308 -0.22603568  1.43583691 -0.96991551 -0.17767864  0.11817785
 -0.11716706  1.          0.          0.28761053  0.06749298  1.47175252
 -0.03480848  0.99132437 -0.06687491  0.10766561  1.          0.        ]


2026-07-21 13:16:46.794 | INFO     |  Action [ 0.2476287  -0.23694094  1.46340239 -0.98013556 -0.14743891  0.12568149
 -0.04242903  1.          0.          0.29474714  0.07889979  1.4754281
  0.0335472   0.9925909  -0.06565073  0.09658036  1.          0.        ]
2026-07-21 13:16:47.003 | INFO     |  Action [ 0.23794372 -0.22535345  1.42098582 -0.95849633 -0.202181    0.11972363
 -0.16147397  1.          0.          0.28507724  0.07050021  1.47471416
  0.00356621  0.99413836 -0.02357977  0.10545252  1.          0.        ]
2026-07-21 13:16:47.841 | INFO     |  Action [ 0.23054139 -0.25260761  1.47221589 -0.98765111 -0.05854515  0.12867233
 -0.06753669  1.          0.          0.26951787  0.07360943  1.47706914
 -0.16906676  0.97934216 -0.02363589  0.10838261  1.          0.        ]
2026-07-21 13:16:48.310 | INFO     |  Action [ 0.22662236 -0.23390776  1.44468284 -0.96299583 -0.1123834   0.11868787
 -0.21429449  1.          0.          0.28501499  0.06749298  1.47706914
 -0.0122176   0

2026-07-21 13:16:51.214 | INFO     |  Action [ 0.22699565 -0.2474532   1.47221589 -0.98097289 -0.09424036  0.11547198
 -0.12440699  1.          0.          0.27045742  0.07480633  1.47706914
 -0.0481805   0.99062443 -0.07307838  0.10488705  1.          0.        ]


...

2026-07-21 13:18:00.062 | INFO     |  Action [ 0.17942317 -0.16665354  0.90524614 -0.30899632  0.93280602  0.04900229
 -0.17886595  1.          0.          0.22790734  0.1188764   0.9663974
  0.00351848  0.99917388 -0.04023983  0.00449008  1.          0.        ]


2026-07-21 13:18:00.294 | INFO     |  Action [ 0.19508454 -0.18174981  0.93599015 -0.21685876  0.97514111  0.04475177
 -0.00832719  1.          0.          0.21619378  0.12411106  0.93014282
 -0.08344334  0.99642026 -0.01334977  0.00237186  1.          0.        ]


2026-07-21 13:18:00.542 | INFO     |  Action [ 0.18379088 -0.17527977  0.89223617 -0.29475611  0.94803804  0.03999819
 -0.11288424  1.          0.          0.23802617  0.11681471  0.9490788
 -0.02599563  0.99785042 -0.06006297  0.00335498  1.          0.        ]


2026-07-21 13:18:00.802 | INFO     |  Action [ 1.87982574e-01 -1.80671573e-01  9.30408299e-01 -2.12141633e-01
  9.72463846e-01  4.15204763e-02  8.70984420e-02  1.00000000e+00
  0.00000000e+00  2.24987075e-01  1.26619890e-01  9.57517684e-01
 -3.94203626e-02  9.97474432e-01  5.90790138e-02  6.58468343e-04
  1.00000000e+00  0.00000000e+00]


2026-07-21 13:18:01.101 | INFO     |  Action [ 0.17772307 -0.16912782  0.87181896 -0.40611365  0.90187758  0.05183186
 -0.13784772  1.          0.          0.25004867  0.11439793  0.93958354
 -0.03972547  0.99913424 -0.00997384  0.00728072  1.          0.        ]
architecture: joint
steps: 200
total_reward: 0.0


In [8]:
run_episode("leader_follower")

2026-07-21 13:18:05.431 | INFO     |  Action [ 2.47305244e-01 -2.43221283e-01  1.46650028e+00 -9.92424548e-01
 -8.18185601e-03  1.22580752e-01  7.94843771e-04  1.00000000e+00
  0.00000000e+00  3.09403360e-01  6.82336017e-02  1.45928371e+00
 -5.25355153e-02  9.90129232e-01  3.22734900e-02  1.25866383e-01
  1.00000000e+00  0.00000000e+00]
2026-07-21 13:18:05.725 | INFO     |  Action [ 0.24907684 -0.2430605   1.46321213 -0.99237174 -0.0168599   0.12204368
  0.00438994  1.          0.          0.30646047  0.06841584  1.46192241
 -0.04459539  0.99058312  0.04593029  0.12102364  1.          0.        ]
2026-07-21 13:18:05.840 | INFO     |  Action [ 2.47534156e-01 -2.43014321e-01  1.46109331e+00 -9.92505670e-01
 -2.06075143e-02  1.20443106e-01  1.21326814e-03  1.00000000e+00
  0.00000000e+00  3.08934480e-01  6.81444183e-02  1.45920289e+00
 -3.30725498e-02  9.91366029e-01  2.63356008e-02  1.24121599e-01
  1.00000000e+00  0.00000000e+00]
2026-07-21 13:18:05.985 | INFO     |  Action [ 0.24747549

In [9]:
tapas_env.close()

[CoppeliaSim:loadinfo]   done.
